# Landslide Prediction Machine Learning Project

This notebook covers the end-to-end machine learning pipeline for predicting landslides using sensor data from two locations: **Veeramalakunnu** and **Thekkil**.

## Objectives:
1. Load and combine sensor data (30-degree and 60-degree datasets).
2. Preprocess data (handle missing values, duplicates, and encoding).
3. Perform Exploratory Data Analysis (EDA) with detailed explanations.
4. Train and compare 4 models: ANN, Decision Tree, Random Forest, and KNN.
5. Evaluate and save the best model for deployment.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Setup paths
DATA_DIR = r"c:\Users\annaa\Downloads\Landslide"
STATIC_DIR = os.path.join(DATA_DIR, "static")
MODEL_DIR = os.path.join(DATA_DIR, "models")

if not os.path.exists(STATIC_DIR):
    os.makedirs(STATIC_DIR)
if not os.path.exists(MODEL_DIR):
    os.makedirs(MODEL_DIR)

### 1. Data Loading and Combination
We load separate Excel files for each slope angle and merge them into a single dataset per location.

In [ ]:
def load_and_combine(place_name, file_configs):
    combined_data = []
    for file_name, angle in file_configs:
        file_path = os.path.join(DATA_DIR, file_name)
        if os.path.exists(file_path):
            df = pd.read_excel(file_path)
            # Add Angle column based on the specific file
            df['Angle'] = angle
            combined_data.append(df)
    
    final_df = pd.concat(combined_data, ignore_index=True)
    return final_df

# Veeramalakunnu
df_veera = load_and_combine("Veeramalakunnu", [("place1_30.xlsx", 30), ("place1_60.xlsx", 60)])

# Thekkil
df_thekkil = load_and_combine("Thekkil", [("Place2_30.xlsx", 30), ("Place2_60.xlsx", 60)])

print(f"Veeramalakunnu Shape: {df_veera.shape}")
print(f"Thekkil Shape: {df_thekkil.shape}")

### 2. Preprocessing
We remove irrelevant columns (Date/Time), handle missing values, and remove duplicates to ensure high-quality data for training.

In [ ]:
def preprocess_data(df):
    # 1. Drop Date/Time
    cols_to_drop = [col for col in df.columns if 'date' in col.lower() or 'time' in col.lower()]
    df = df.drop(columns=cols_to_drop)
    
    # 2. Handle missing values
    df = df.dropna()
    
    # 3. Handle duplicates
    df = df.drop_duplicates()
    
    return df

df_veera = preprocess_data(df_veera)
df_thekkil = preprocess_data(df_thekkil)

### 3. Exploratory Data Analysis (EDA)
Detailed visualizations to understand distribution and relationships.

In [ ]:
def perform_eda(df, place_name):
    print(f"--- EDA for {place_name} ---")
    print(f"Columns: {df.columns.tolist()}")
    print(f"Null Check:\n{df.isnull().sum()}")
    
    # Category Distribution
    plt.figure(figsize=(8,5))
    sns.countplot(x='Category', data=df)
    plt.title(f'Category Distribution - {place_name}')
    plt.show()
    
    # Correlation Heatmap
    plt.figure(figsize=(10,8))
    num_cols = df.select_dtypes(include=[np.number]).columns
    sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm')
    plt.title(f'Correlation Heatmap - {place_name}')
    plt.show()

perform_eda(df_veera, "Veeramalakunnu")
perform_eda(df_thekkil, "Thekkil")

### 4. Training and Evaluation
We train 4 models and select the best one based on test accuracy.

In [ ]:
# This step is handled in the background by our train_models.py script for efficiency.
# It generates accuracy tables, confusion matrices, and saved model files.
print("Training models for both locations...")
# Summary table for Veeramalakunnu
veera_results = pd.read_csv(os.path.join(DATA_DIR, 'Veeramalakunnu_comparison.csv'))
display(veera_results)

veera_best = veera_results.loc[veera_results['Accuracy'].idxmax()]['Model']
print(f"Best model for Veeramalakunnu: {veera_best}")